# TTFT와 total 비교

같은 에이전트를 세 모드로 실행할 때 첫 사건, 첫 토큰, 전체 완료 시간은 어떻게 달라질까요?


## 실행 환경 준비

Python 3.12와 고정된 패키지 환경을 사용합니다. 키 값은 출력하지 않고 존재 여부만 확인합니다.


In [1]:
# 고정한 패키지를 설치하고 비밀 키의 존재 여부만 확인합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3
from pathlib import Path
import os
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break

MODEL = "gpt-5.4-mini"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print({"langchain": version("langchain"), "langgraph": version("langgraph"), "model": MODEL, "cost_limit_usd": 1.0})


Note: you may need to restart the kernel to use updated packages.
{'langchain': '1.4.0', 'langgraph': '1.2.11', 'model': 'gpt-5.4-mini', 'cost_limit_usd': 1.0}


## 측정 그래프 준비


In [2]:
# 같은 모델과 질문을 세 모드에서 측정할 그래프를 구성합니다.
from time import perf_counter
from typing import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    answer: str

model = init_chat_model(f"openai:{MODEL}", reasoning_effort="none")
def answer(state: State) -> dict[str, str]:
    get_stream_writer()({"stage": "started"})
    response = model.invoke("세 문장 이내로 답하세요: " + state["question"])
    return {"answer": response.text}

builder = StateGraph(State)
builder.add_node("answer", answer)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile()


## 세 모드에서 시간 측정


In [3]:
# first_event와 내용이 있는 첫 토큰, 마지막 사건 시각을 따로 기록합니다.
question = "고객에게 스트리밍의 장점을 설명해 주세요."
rows = []
for mode in ["updates", "messages", "custom"]:
    started = perf_counter()
    first_event = None
    first_token = None
    event_count = 0
    for data in graph.stream({"question": question, "answer": ""}, stream_mode=mode):
        elapsed = perf_counter() - started
        event_count += 1
        first_event = elapsed if first_event is None else first_event
        if mode == "messages":
            chunk, _ = data
            if chunk.text and first_token is None:
                first_token = elapsed
    total = perf_counter() - started
    rows.append({"mode": mode, "first_event_s": round(first_event or total, 3), "ttft_s": None if first_token is None else round(first_token, 3), "total_s": round(total, 3), "events": event_count})
for row in rows:
    print(row)
assert next(r for r in rows if r["mode"] == "messages")["ttft_s"] is not None


{'mode': 'updates', 'first_event_s': 1.183, 'ttft_s': None, 'total_s': 1.183, 'events': 1}
{'mode': 'messages', 'first_event_s': 1.109, 'ttft_s': 1.11, 'total_s': 1.478, 'events': 65}
{'mode': 'custom', 'first_event_s': 0.001, 'ttft_s': None, 'total_s': 1.231, 'events': 1}


## 지표를 구분해 해석


In [4]:
# TTFT가 있는 messages와 첫 사건만 있는 두 모드를 구분합니다.
for row in rows:
    meaning = "첫 토큰을 직접 측정" if row["ttft_s"] is not None else "첫 사건만 측정"
    print(row["mode"], meaning, "total=", row["total_s"])
print("네트워크 상태에 따라 수치는 실행할 때마다 달라질 수 있습니다.")


updates 첫 사건만 측정 total= 1.183
messages 첫 토큰을 직접 측정 total= 1.478
custom 첫 사건만 측정 total= 1.231
네트워크 상태에 따라 수치는 실행할 때마다 달라질 수 있습니다.
